In [1]:

import os, json, math, time
from pathlib import Path
from tqdm import tqdm
import numpy as np
from PIL import Image

import torch
import torch.nn as nn
from torchvision import transforms, models




In [ ]:
# PROJECT paths - edit if different
PROJECT_ROOT = Path.home() / "Downloads" / "AI_Tumour_detection" 
ORCHID_ROOT = PROJECT_ROOT / "data" / "ORCHID"
EMB_ROOT = PROJECT_ROOT / "embeddings"

# ensure embeddings root exists
(EMB_ROOT).mkdir(parents=True, exist_ok=True)

# Device: prefer MPS (Apple), else CPU
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print("Using device:", device)



Using device: mps


In [9]:
BATCH_SIZE = 64 
EMB_DIM = 512
IMAGE_SIZE = 224    

In [10]:
def make_feature_extractor(emb_dim=512, device=device):
    # load pretrained resnet50
    resnet = models.resnet50(pretrained=True)
    # remove final FC: get feature map output (2048) after avgpool
    backbone = nn.Sequential(*list(resnet.children())[:-1])  # outputs (B,2048,1,1)
    # projection head
    proj = nn.Sequential(
        nn.Flatten(),            # (B, 2048)
        nn.Linear(2048, emb_dim),
        nn.ReLU(inplace=True),
    )
    model = nn.Sequential(backbone, proj)
    model.to(device)
    model.eval()
    return model

model = make_feature_extractor(EMB_DIM, device=device)
print(model)


Sequential(
  (0): Sequential(
    (0): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
    (4): Sequential(
      (0): Bottleneck(
        (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
        (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (bn3): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
        (relu): ReLU(inplace=True)
        (downsample): Sequential(
          (0): Conv2d(64, 256

In [11]:
# transforms and batch loader
transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std =[0.229, 0.224, 0.225]),
])

def load_image_tensor(path):
    img = Image.open(path).convert("RGB")
    return transform(img)

def batchify(patch_paths, batch_size=BATCH_SIZE):
    """Yield batches of (paths, tensor)"""
    tensors = []
    paths = []
    for p in patch_paths:
        tensors.append(load_image_tensor(p))
        paths.append(p)
        if len(tensors) == batch_size:
            yield paths, torch.stack(tensors, dim=0)
            tensors, paths = [], []
    if tensors:
        yield paths, torch.stack(tensors, dim=0)


In [12]:
#  iterate slides and save embeddings
from pathlib import Path
import os

# Load slide index CSV we saved earlier
idx_csv = PROJECT_ROOT / "notebooks" / "slide_index.csv"
import pandas as pd
df_index = pd.read_csv(idx_csv)

# progress summary
total_slides = len(df_index)
print(f"Slides to process: {total_slides}, total patches ~ {df_index['n_patches'].sum()}")

start_time = time.time()
for _, row in tqdm(df_index.iterrows(), total=total_slides, desc="Slides"):
    split = row["split"]
    cls = row["class"]
    slide_id = row["slide_id"]
    patch_list = json.loads(row["patch_paths"].replace("'", '"')) if isinstance(row["patch_paths"], str) else row["patch_paths"]

    # prepare save dirs
    out_dir = EMB_ROOT / split / cls
    out_dir.mkdir(parents=True, exist_ok=True)
    out_npy = out_dir / f"{slide_id}.npy"
    out_meta = out_dir / f"{slide_id}_meta.json"

    # skip if already exists
    if out_npy.exists() and out_meta.exists():
        continue

    emb_list = []
    ordered_patch_names = []

    # process in batches
    for paths_batch, tensor_batch in batchify(patch_list, batch_size=BATCH_SIZE):
        tensor_batch = tensor_batch.to(device)
        with torch.no_grad():
            feats = model(tensor_batch)  # shape (B, EMB_DIM)
            feats = feats.cpu().numpy()
        emb_list.append(feats)
        ordered_patch_names.extend(paths_batch)

    # concat and save
    if len(emb_list) > 0:
        emb_all = np.concatenate(emb_list, axis=0)  # (n_patches, EMB_DIM)
    else:
        # rare: no patches (shouldn't happen) -> skip
        continue

    # final sanity check: shape matches n_patches
    assert emb_all.shape[0] == len(ordered_patch_names), "Mismatch lengths"

    # save .npy and meta json
    np.save(out_npy, emb_all.astype(np.float32))
    meta = {"patch_paths": ordered_patch_names, "n_patches": emb_all.shape[0], "emb_dim": emb_all.shape[1]}
    with open(out_meta, "w") as f:
        json.dump(meta, f)

# summary time
elapsed = time.time() - start_time
print(f"Done. Time elapsed: {elapsed/60:.2f} minutes")


Slides to process: 448, total patches ~ 14705


Slides: 100%|██████████| 448/448 [09:56<00:00,  1.33s/it]

Done. Time elapsed: 9.94 minutes
